# ABS Labour Force Survey data

The notebook’s goal is to automate the monthly preparation of ABS Labour Force data for analysis and dashboards. It identifies the next release to process, downloads three workbooks covering national and state labour force figures, youth figures, and modelled regional estimates, then checks them against reference files for structural or definition changes. It reshapes the data into consistent, clearly labelled tables and saves three cleaned CSVs, while retaining the original downloads and a source manifest so each completed run is traceable.

- **Table 010:** Labour force status by Sex, State and Territory — Trend, Seasonally adjusted and Original
- **Table 014:** Labour force status for 15–24 year olds by State and Territory
- **MLF1:** Modelled estimates of Labour force status by SA4 (ASGS), Age and Sex

**Outputs:** three cleaned CSV files named for their source table and download month, plus a source manifest, in a new dated folder under `outputs/abs_labour_force/`. Downloads are retained under `raw/` in that folder, written only once all three have downloaded. The source manifest is written only after all three clean files are saved; earlier successful runs are kept separately.

### What is a material change?

| Stop the run | Allow without changing the baseline |
|---|---|
| Missing or additional sheets; moved header rows; changed MLF1 table identity | New monthly observations |
| Added, removed or duplicate series IDs; changed descriptions, units, frequency, series type, or historical start | Revisions to numeric estimates |
| Changed MLF1 columns, age/sex categories, or region codes/names (including geography vintage) | Reordered series columns within a data sheet; reordered MLF1 rows |
| Blank or invalid estimates in MLF1; MLF1 months with differing row counts | Cosmetic formatting; copyright and release-date updates |
| Latest month in any workbook is not the release being processed | |

## 1. Setup and reference workbooks

Use Python 3.10+ with `pandas`, `openpyxl`, `requests` and a Jupyter kernel. In a new environment, run `%pip install pandas openpyxl requests` once, then restart the kernel. Paths assume the notebook is opened from the project folder.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import requests
from helper import validate_observations
from IPython.display import display

# Separate incase the base-url for any dataset changes in the future.
base_urls = {
    "Table 010": "https://www.abs.gov.au/statistics/labour/employment-and-unemployment/labour-force-australia/",
    "Table 014": "https://www.abs.gov.au/statistics/labour/employment-and-unemployment/labour-force-australia/",
    "MLF1": "https://www.abs.gov.au/statistics/labour/employment-and-unemployment/labour-force-australia/",
}

root = Path.cwd()
baseline = root / "Labour Force Datasets"
now = datetime.now(timezone.utc)
retrieved_at = now.isoformat()
run = root / "outputs" / "abs_labour_force" / now.strftime("%Y%m%dT%H%M%S%fZ")

## 2. Build the next release URLs and download all three files

The next release is one month after the later of the last month in the reference Table 010 workbook and the latest completed run. For the three workbooks, build the observed ABS URL pattern from that month and each fixed filename. The download fails if a file is unavailable. All three are held in memory and the run folder is created only after every download succeeds, so trying an unpublished release leaves nothing behind. MLF1 is scheduled to move to a separate ABS publication from October 2026, so its URL will need updating then.

In [ ]:
sources = pd.DataFrame(
    [("Table 010", "Table 010.xlsx", "62020010.xlsx"), ("Table 014", "Interim Table 14.xlsx", "62020014.xlsx"), ("MLF1", "MLF1.xlsx", "MLF1.xlsx")],
    columns=["dataset", "baseline_file", "download_file"],
).set_index("dataset")

baseline_file = baseline / sources.loc["Table 010", "baseline_file"]
baseline_latest_month = pd.to_datetime(pd.read_excel(baseline_file, sheet_name="Data1", header=None).iloc[-1, 0])
completed_sources = (root / "outputs" / "abs_labour_force").glob("*/sources.csv")
processed_months = [pd.to_datetime(pd.read_csv(source_file)["reference_period"]).max() for source_file in completed_sources]
download_month = max([baseline_latest_month, *processed_months]) + pd.offsets.MonthBegin(1)

sources["source_url"] = sources.index.map(base_urls) + f"{download_month:%b-%Y}/".lower() + sources["download_file"]
sources["reference_period"] = download_month
sources["retrieved_at_utc"] = retrieved_at

downloads = {}

for row in sources.itertuples():
    response = requests.get(row.source_url, timeout=120, headers={"Cache-Control": "no-cache"})
    response.raise_for_status()
    assert response.content.startswith(b"PK"), f"{row.Index}: download is not an XLSX file"
    downloads[row.download_file] = response.content

(run / "raw").mkdir(parents=True)
for name, content in downloads.items():
    (run / "raw" / name).write_bytes(content)

print("Download month:", f"{download_month:%B %Y}")
display(sources[["download_file", "reference_period", "source_url"]])

## 3. Read and validate Table 010

This table has its own workbook reads, layout settings, validation and description parsing. Read every populated data column; expected sheets and header labels come from its reference workbook. The seven series-definition fields must match by ABS series ID, and the latest source month must be the requested download month.

Both time-series tables use the same observation checks described below.

`validate_observations` in `helper.py` checks preservation against the original downloaded sheet: the date/series keys and their counts must survive reshaping, including repeated keys. The number of populated observations for each date/series key must also match. Repeated occurrences of the same key are counted together, not distinguished individually. Existing source blanks are allowed without a warning. Nonnumeric or infinite values are displayed and stop the run; missing values follow pandas’ existing Excel-reading defaults.

The metadata join must preserve the observation keys and their counts, and `many_to_one` prevents duplicate metadata matches. These checks detect lost or duplicated observations and changed population of values; they do not compare individual numeric values or assess the source’s completeness against `n_obs` or its published start/end dates.


### Read Table 010

In [ ]:
row = sources.loc["Table 010"]
table_010_base = pd.read_excel(baseline / row["baseline_file"], sheet_name=None, header=None)
table_010_book = pd.read_excel(run / "raw" / row["download_file"], sheet_name=None, header=None)
print("Table 010 sheets:", list(table_010_book))

### Validate and reshape Table 010

Excel rows 1–10 contain metadata; monthly observations start at row 11. Keep this table’s row positions and definition fields here so changes to its layout can be made locally.

In [ ]:
dataset = "Table 010"
assert set(table_010_book) == set(table_010_base), f"{dataset}: workbook sheets changed"

metadata_fields = ["description", "unit", "series_type", "data_type", "frequency", "collection_month", "series_start", "series_end", "n_obs", "series_id"]

# Excel rows 1–10 contain metadata; monthly data start at row 11.
# Row positions in pandas are zero-based.
first_data_row = len(metadata_fields)
series_id_row = metadata_fields.index("series_id")
# New releases can extend end dates and observation counts, so compare
# only the series-definition fields with the baseline.
definition_fields = ["description", "unit", "series_type", "data_type", "frequency", "collection_month", "series_start"]

metadata_frames, observation_frames, sheet_checks = [], [], []
for sheet, raw_sheet in table_010_book.items():
    if not sheet.startswith("Data"):
        continue
    baseline_sheet = table_010_base[sheet]
    pd.testing.assert_series_equal(raw_sheet.iloc[:first_data_row, 0], baseline_sheet.iloc[:first_data_row, 0], obj=f"{dataset}/{sheet}: header rows")
    series_ids = raw_sheet.iloc[series_id_row, 1:]
    assert series_ids.notna().all() and series_ids.is_unique, f"{dataset}/{sheet}: missing or duplicate series IDs"

    # Align by series ID so column reordering does not cause a
    # definition mismatch.
    sheet_metadata = raw_sheet.iloc[:first_data_row, 1:].T.set_axis(metadata_fields, axis=1).set_index("series_id").sort_index()
    baseline_metadata = baseline_sheet.iloc[:first_data_row, 1:].T.set_axis(metadata_fields, axis=1).set_index("series_id").sort_index()
    pd.testing.assert_frame_equal(sheet_metadata[definition_fields], baseline_metadata[definition_fields], obj=f"{dataset}/{sheet}: series definitions")

    sheet_observations = (
        raw_sheet.iloc[first_data_row:].set_axis(["date", *series_ids], axis=1).melt("date", var_name="series_id").astype({"date": "datetime64[ns]"})
    )
    sheet_observations = validate_observations(raw_sheet, sheet_observations, first_data_row, dataset, sheet, download_month)
    observation_frames.append(sheet_observations.assign(dataset=dataset))
    metadata_frames.append(sheet_metadata.reset_index().assign(dataset=dataset, sheet=sheet))
    sheet_checks.append((dataset, sheet, sheet_observations["date"].nunique(), len(sheet_metadata), len(sheet_observations), sheet_observations["date"].max()))

table_010_metadata = pd.concat(metadata_frames, ignore_index=True)
table_010 = pd.concat(observation_frames, ignore_index=True)
assert not table_010_metadata.duplicated(["dataset", "series_id"]).any(), f"{dataset}: a series ID is repeated across sheets"
display(pd.DataFrame(sheet_checks, columns=["dataset", "sheet", "months", "series", "observations", "latest_month"]))

### Tidy Table 010 dimensions

Assumption: Table 010 covers people aged 15 years and over. Its descriptions list measure, sex and region. Remove ABS hierarchy markers (`>`) only after validating the original descriptions. Keep published units: `000` means thousands of people and `Percent` remains a percentage (4.5 means 4.5%).

In [ ]:
description_parts = table_010_metadata["description"].str.strip().str.rstrip(";").str.split(";", expand=True)
description_parts = description_parts.apply(lambda s: s.str.strip().str.lstrip(">").str.strip())
table_010_metadata["measure"] = description_parts[0]
table_010_metadata["sex"] = description_parts[1]
table_010_metadata["region"] = description_parts[2]
table_010_metadata["age"] = "15 years and over"
assert table_010_metadata[["measure", "sex", "region", "age"]].notna().all().all()

# Compare key counts so the join cannot silently drop observations.
# Counting repeated keys also preserves their multiplicity.
observation_keys = ["dataset", "date", "series_id"]
keys_before_join = table_010[observation_keys].value_counts(sort=False).sort_index()
table_010 = table_010.merge(
    table_010_metadata[["dataset", "series_id", "measure", "sex", "age", "region", "series_type", "data_type", "unit"]],
    on=["dataset", "series_id"],
    validate="many_to_one",
)
pd.testing.assert_series_equal(
    keys_before_join, table_010[observation_keys].value_counts(sort=False).sort_index(), obj="Table 010 metadata join: observation keys and counts"
)
print(table_010.shape)
display(table_010[["date", "series_id", "measure", "region", "sex", "age", "series_type", "data_type", "unit", "value"]].head())

## 4. Read and validate Table 014

This table has its own workbook reads, layout settings, validation and description parsing. Read every populated data column; expected sheets and header labels come from its reference workbook. The seven series-definition fields must match by ABS series ID, and the latest source month must be the requested download month.

### Read Table 014

In [ ]:
row = sources.loc["Table 014"]
table_014_base = pd.read_excel(baseline / row["baseline_file"], sheet_name=None, header=None)
table_014_book = pd.read_excel(run / "raw" / row["download_file"], sheet_name=None, header=None)
print("Table 014 sheets:", list(table_014_book))

### Validate and reshape Table 014

Excel rows 1–10 contain metadata; monthly observations start at row 11. Keep this table’s row positions and definition fields here so changes to its layout can be made locally.

In [ ]:
dataset = "Table 014"
assert set(table_014_book) == set(table_014_base), f"{dataset}: workbook sheets changed"

metadata_fields = ["description", "unit", "series_type", "data_type", "frequency", "collection_month", "series_start", "series_end", "n_obs", "series_id"]

# Excel rows 1–10 contain metadata; monthly data start at row 11.
# Row positions in pandas are zero-based.
first_data_row = len(metadata_fields)
series_id_row = metadata_fields.index("series_id")
# New releases can extend end dates and observation counts, so compare
# only the series-definition fields with the baseline.
definition_fields = ["description", "unit", "series_type", "data_type", "frequency", "collection_month", "series_start"]

metadata_frames, observation_frames, sheet_checks = [], [], []
for sheet, raw_sheet in table_014_book.items():
    if not sheet.startswith("Data"):
        continue
    baseline_sheet = table_014_base[sheet]
    pd.testing.assert_series_equal(raw_sheet.iloc[:first_data_row, 0], baseline_sheet.iloc[:first_data_row, 0], obj=f"{dataset}/{sheet}: header rows")
    series_ids = raw_sheet.iloc[series_id_row, 1:]
    assert series_ids.notna().all() and series_ids.is_unique, f"{dataset}/{sheet}: missing or duplicate series IDs"

    # Align by series ID so column reordering does not cause a
    # definition mismatch.
    sheet_metadata = raw_sheet.iloc[:first_data_row, 1:].T.set_axis(metadata_fields, axis=1).set_index("series_id").sort_index()
    baseline_metadata = baseline_sheet.iloc[:first_data_row, 1:].T.set_axis(metadata_fields, axis=1).set_index("series_id").sort_index()
    pd.testing.assert_frame_equal(sheet_metadata[definition_fields], baseline_metadata[definition_fields], obj=f"{dataset}/{sheet}: series definitions")

    sheet_observations = (
        raw_sheet.iloc[first_data_row:].set_axis(["date", *series_ids], axis=1).melt("date", var_name="series_id").astype({"date": "datetime64[ns]"})
    )
    sheet_observations = validate_observations(raw_sheet, sheet_observations, first_data_row, dataset, sheet, download_month)
    observation_frames.append(sheet_observations.assign(dataset=dataset))
    metadata_frames.append(sheet_metadata.reset_index().assign(dataset=dataset, sheet=sheet))
    sheet_checks.append((dataset, sheet, sheet_observations["date"].nunique(), len(sheet_metadata), len(sheet_observations), sheet_observations["date"].max()))

table_014_metadata = pd.concat(metadata_frames, ignore_index=True)
table_014 = pd.concat(observation_frames, ignore_index=True)
assert not table_014_metadata.duplicated(["dataset", "series_id"]).any(), f"{dataset}: a series ID is repeated across sheets"
display(pd.DataFrame(sheet_checks, columns=["dataset", "sheet", "months", "series", "observations", "latest_month"]))

### Tidy Table 014 dimensions

Assumption: Table 014 covers persons aged 15–24. Its descriptions list region then measure; there is no sex split. Remove ABS hierarchy markers (`>`) only after validating the original descriptions. Keep published units: `000` means thousands of people and `Percent` remains a percentage (4.5 means 4.5%).

In [ ]:
description_parts = table_014_metadata["description"].str.strip().str.rstrip(";").str.split(";", expand=True)
description_parts = description_parts.apply(lambda s: s.str.strip().str.lstrip(">").str.strip())
table_014_metadata["region"] = description_parts[0]
table_014_metadata["measure"] = description_parts[1]
table_014_metadata["sex"] = "Persons"
table_014_metadata["age"] = "15-24 years"
assert table_014_metadata[["measure", "sex", "region", "age"]].notna().all().all()

# Compare key counts so the join cannot silently drop observations.
# Counting repeated keys also preserves their multiplicity.
observation_keys = ["dataset", "date", "series_id"]
keys_before_join = table_014[observation_keys].value_counts(sort=False).sort_index()
table_014 = table_014.merge(
    table_014_metadata[["dataset", "series_id", "measure", "sex", "age", "region", "series_type", "data_type", "unit"]],
    on=["dataset", "series_id"],
    validate="many_to_one",
)
pd.testing.assert_series_equal(
    keys_before_join, table_014[observation_keys].value_counts(sort=False).sort_index(), obj="Table 014 metadata join: observation keys and counts"
)
print(table_014.shape)
display(table_014[["date", "series_id", "measure", "region", "sex", "age", "series_type", "data_type", "unit", "value"]].head())

## 5. Read and validate MLF1 regional estimates

Use the underlying **Data 1** sheet, which contains all sex × age × region combinations. The presentation pivot sheet is not another dataset and is not appended. Ignore only completely empty columns below the workbook preamble; any populated new header or data column is compared with the baseline. Every month must have the same number of rows, and the latest month must be the release being processed.

### Read MLF1

In [ ]:
row = sources.loc["MLF1"]
baseline_workbook = pd.ExcelFile(baseline / row["baseline_file"])
downloaded_workbook = pd.ExcelFile(run / "raw" / row["download_file"])

mlf_base = pd.read_excel(baseline_workbook, sheet_name="Data 1", header=None)
mlf_raw = pd.read_excel(downloaded_workbook, sheet_name="Data 1", header=None)
print("MLF1 sheets:", downloaded_workbook.sheet_names)

### Validate MLF1

In [ ]:
assert set(downloaded_workbook.sheet_names) == set(baseline_workbook.sheet_names), "MLF1: workbook sheets changed"

# The table title is in Excel cell A2.
assert mlf_raw.iloc[1, 0] == mlf_base.iloc[1, 0], "MLF1: table identity changed"

# Excel row 4 contains the headings after the preamble.
# Its zero-based position is 3.
mlf_header_row = 3
mlf_base = mlf_base.iloc[mlf_header_row:].dropna(axis=1, how="all")
mlf_raw = mlf_raw.iloc[mlf_header_row:].dropna(axis=1, how="all")
assert mlf_raw.iloc[0].tolist() == mlf_base.iloc[0].tolist(), "MLF1: column names, units, order or geography vintage changed"
mlf_base = mlf_base.iloc[1:].set_axis(mlf_base.iloc[0].tolist(), axis=1).reset_index(drop=True).infer_objects()
mlf_raw = mlf_raw.iloc[1:].set_axis(mlf_raw.iloc[0].tolist(), axis=1).reset_index(drop=True).infer_objects()

# Columns after Month identify sex, age and SA4 region, in that order.
dimension_columns = mlf_raw.columns[1:4].tolist()
measure_columns = mlf_raw.columns[4:].tolist()
region_column = dimension_columns[2]
assert mlf_raw.notna().all().all(), "MLF1: missing dimension or estimate"
assert mlf_raw[measure_columns].dtypes.map(pd.api.types.is_numeric_dtype).all(), "MLF1: non-numeric observations"
for column in dimension_columns:
    assert set(mlf_raw[column]) == set(mlf_base[column]), f"MLF1: categories changed in {column}"

mlf_raw["Month"] = pd.to_datetime(mlf_raw["Month"])
rows_per_month = mlf_raw.groupby("Month").size()
assert rows_per_month.nunique() == 1, "MLF1: months have different numbers of sex × age × region rows"
latest_month = mlf_raw["Month"].max()
assert latest_month == download_month, f"MLF1: latest month is not {download_month:%B %Y}"

print(mlf_raw.shape)
display(mlf_raw.head())
display(
    pd.DataFrame(
        {
            "months": [len(rows_per_month)],
            "regions": [mlf_raw[region_column].nunique()],
            "sex_categories": [mlf_raw["Sex"].nunique()],
            "age_groups": [mlf_raw["Age"].nunique()],
            "rows_per_month": [rows_per_month.iloc[0]],
            "latest_month": [latest_month],
        }
    )
)

### Tidy regional measures

The workbook labels its geography **Labour market region (SA4): ASGS (2011)**. Preserve that definition and the published region names, including combined regions. `series_type = Modelled` identifies this source; it does not assert a seasonal adjustment status. MLF1 does not supply ABS time-series IDs, so no series ID is exported. Only the underlying Data 1 regional records are used; national and state totals from the presentation pivot are not imported. No totals, rates or additional populations are inferred from the pivot table.

In [ ]:
mlf = mlf_raw.rename(columns={"Month": "date", "Sex": "sex", "Age": "age", region_column: "region"}).melt(
    id_vars=["date", "sex", "age", "region"], value_vars=measure_columns, var_name="measure"
)
mlf[["region_code", "region"]] = mlf["region"].str.extract(r"^(\d{3}) (.+)$")
assert mlf["region_code"].notna().all(), "MLF1: region-code format changed"
mlf["measure"] = mlf["measure"].str.replace(" ('000)", "", regex=False)
mlf["unit"] = "000"
mlf["series_type"] = "Modelled"
print(mlf.shape)
display(mlf[["date", "region_code", "region", "sex", "age", "measure", "value"]].head())

## 6. Save three cleaned source tables

Each source workbook gets its own cleaned CSV named for the ABS table and download month, for example `table_010_2026-08.csv`. The release month is recorded in the filename and in `sources.csv`, not as a column.

| File | Columns |
|---|---|
| `table_010_*.csv`, `table_014_*.csv` | `date`, `series_id`, `measure`, `sex`, `age`, `region`, `series_type`, `data_type`, `unit`, `value` |
| `mlf1_*.csv` | `date`, `measure`, `sex`, `age`, `sa4_code`, `sa4_region`, `series_type`, `unit`, `value` |

`region` in Tables 010 and 014 is Australia or a state/territory. `sources.csv` records each ABS download and its output file; it is written last, so it only exists for completed runs.

In [ ]:
export_columns = ["date", "series_id", "measure", "sex", "age", "region", "series_type", "data_type", "unit", "value"]
exports = {"Table 010": table_010[export_columns], "Table 014": table_014[export_columns]}
exports["MLF1"] = mlf[["date", "measure", "sex", "age", "region_code", "region", "series_type", "unit", "value"]].rename(
    columns={"region_code": "sa4_code", "region": "sa4_region"}
)

for dataset, dataset_rows in exports.items():
    sources.loc[dataset, "output_file"] = f"{dataset.lower().replace(' ', '_')}_{download_month:%Y-%m}.csv"
    dataset_rows.to_csv(run / sources.loc[dataset, "output_file"], index=False, date_format="%Y-%m-%d")

# Write the manifest after all exports so it marks a completed run.
sources.to_csv(run / "sources.csv")
print(f"Saved {len(exports)} cleaned CSV files for {download_month:%B %Y} in {run}")
display(sources[["download_file", "output_file"]])

with pd.option_context("display.max_columns", None):
    for dataset, dataset_rows in exports.items():
        print(f"\n{sources.loc[dataset, 'output_file']} — {dataset_rows.shape[0]:,} rows × {dataset_rows.shape[1]} columns")
        display(dataset_rows.head())